In [ ]:
import os
import pandas as pd
import shutil
import os
from glob import glob
import logging

from onekey_algo import get_param_in_cwd
from onekey_algo.utils import create_img_msk_dir
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.Radiology import get_image_mask_from_dir, diagnose_3d_image_mask_settings

root = get_param_in_cwd('radio_dir')
label_data = pd.read_csv(os.path.join(root, 'Survival.csv'))
label_data['group'] = 'train'
display(label_data['group'].value_counts())
label_data

In [ ]:
for mask in glob(os.path.join(root, 'masks_PDAC_nii', '*/tumor.nii.gz')):
    sname = os.path.dirname(mask).split('_')[-1]
    shutil.copy(mask, os.path.join(root, 'masks', f'{sname}.nii.gz'))

In [ ]:
set(os.listdir(os.path.join(root, 'masks'))) - set(os.listdir(os.path.join(root, 'images')))

In [ ]:
import os
import re
import shutil
import pandas as pd
from onekey_algo.custom.components.comp2 import split_dataset4sol
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.utils import print_join_info

samples = pd.DataFrame(os.listdir(os.path.join(root, 'images')), columns=['ID'])
print_join_info(samples, label_data)
label_data = pd.merge(label_data, samples['ID'], on='ID', how='inner')
train_data = label_data[label_data['group'] == 'train']
test_data = label_data[label_data['group'] == 'test']
rt = split_dataset4sol(train_data, cv=False, n_trails=1, test_size=0.3, save_dir='.', shuffle=True, map_ext=False)
for idx, (train, val) in enumerate(rt):
#     val['group'] = 'val'
    rnd = pd.concat([train, val, test_data], axis=0)
    display(rnd['group'].value_counts())
    rnd.to_csv(f'split_info/label-RND-{idx}.csv', index=False)
#     rnd[rnd['group'] == 'train'][['ID', 'label']].to_csv(f'split_info/train3d-RND-{idx}.txt', sep='\t', header=None, index=None)
#     rnd[rnd['group'] != 'train'][['ID', 'label']].to_csv(f'split_info/val3d-RND-{idx}.txt', sep='\t', header=None, index=None)
    
#     pd.merge(samples, rnd[rnd['group'] == 'train'], on='ID', how='inner')[['fname', 'label']].to_csv(f'split_info/train25d-RND-{idx}.txt', 
#                                                                                                      sep='\t', header=None, index=None)
#     pd.merge(samples, rnd[rnd['group'] != 'train'], on='ID', how='inner')[['fname', 'label']].to_csv(f'split_info/val25d-RND-{idx}.txt', 
#                                                                                                      sep='\t', header=None, index=None)
    
#     rnd['ID'] = rnd['ID'].map(lambda x: x.replace('.nii.gz', '.nii.npy'))
#     rnd[rnd['group'] == 'train'][['ID', 'label']].to_csv(f'split_info/train2d-RND-{idx}.txt', sep='\t', header=None, index=None)
#     rnd[rnd['group'] != 'train'][['ID', 'label']].to_csv(f'split_info/val2d-RND-{idx}.txt', sep='\t', header=None, index=None)